# 011 Classification Ladder Baselines

This notebook starts the classification ladder with transparent dev-only baselines. It does not train, embed, call an LLM, generate answers, or score holdout.

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OUT = ROOT / 'data' / 'eval' / 'classification_ladder_dev'
SUMMARY_MD = OUT / 'classification_ladder_summary_dev.md'

ROUTING_DEV = ROOT / 'data' / 'eval' / 'candidate_loader_and_eval_splits' / 'routing_eval_dev.jsonl'
SAFETY_DEV = ROOT / 'data' / 'eval' / 'candidate_loader_and_eval_splits' / 'safety_eval_dev.jsonl'
assert ROUTING_DEV.exists(), f'Missing routing dev fixture: {ROUTING_DEV}'
assert SAFETY_DEV.exists(), f'Missing safety dev fixture: {SAFETY_DEV}'

## Build Artifacts

In [ ]:
RUN_ANALYSIS = True

if RUN_ANALYSIS:
    subprocess.run(
        [
            sys.executable,
            str(ROOT / 'scripts' / 'evaluate_classification_ladder_baselines.py'),
        ],
        cwd=ROOT,
        check=True,
    )

assert SUMMARY_MD.exists(), f'Missing classification ladder summary: {SUMMARY_MD}'

## Summary

In [ ]:
display(Markdown(SUMMARY_MD.read_text(encoding='utf-8')))

## Metric Tables

In [ ]:
primary = pd.read_csv(OUT / 'classification_primary_multiclass_metrics_dev.csv')
multilabel = pd.read_csv(OUT / 'classification_multilabel_domain_metrics_dev.csv')
ranked = pd.read_csv(OUT / 'classification_ranked_domain_metrics_dev.csv')
safety_behavior = pd.read_csv(OUT / 'classification_safety_behavior_metrics_dev.csv')
safety_signal = pd.read_csv(OUT / 'classification_safety_signal_metrics_dev.csv')
primary_per_domain = pd.read_csv(OUT / 'classification_primary_per_domain_dev.csv')
multilabel_per_domain = pd.read_csv(OUT / 'classification_multilabel_per_domain_dev.csv')
safety_signal_per_label = pd.read_csv(OUT / 'classification_safety_signal_per_label_dev.csv')

display(primary)
display(multilabel)
display(ranked)
display(safety_behavior)
display(safety_signal)

## Ladder Overview

In [ ]:
overview = primary[['profile', 'accuracy']].rename(columns={'accuracy': 'primary_accuracy'})
overview = overview.merge(multilabel[['profile', 'micro_f1']], on='profile')
overview = overview.rename(columns={'micro_f1': 'multilabel_micro_f1'})
overview = overview.merge(ranked[['profile', 'primary_hit_at_3', 'graded_ndcg_at_3']], on='profile')
overview = overview.merge(safety_behavior[['profile', 'accuracy']], on='profile')
overview = overview.rename(columns={'accuracy': 'safety_behavior_accuracy'})
overview = overview.merge(safety_signal[['profile', 'micro_f1']], on='profile')
overview = overview.rename(columns={'micro_f1': 'safety_signal_micro_f1'})
display(overview)

ax = overview.set_index('profile').plot(kind='bar', figsize=(10, 4), rot=0)
ax.set_title('Transparent classification ladder baselines')
ax.set_xlabel('Profile')
ax.set_ylabel('Metric value')
ax.set_ylim(0, 1.05)
plt.tight_layout()

## Domain Diagnostics

In [ ]:
domain_f1 = primary_per_domain.pivot(index='domain', columns='profile', values='f1')
ax = domain_f1.plot(kind='barh', figsize=(9, 5))
ax.set_title('Primary-domain multi-class F1 by domain')
ax.set_xlabel('F1')
ax.set_ylabel('Domain')
ax.set_xlim(0, 1.05)
plt.tight_layout()

display(primary_per_domain.sort_values(['profile', 'f1']))
display(multilabel_per_domain.sort_values(['profile', 'f1']))

## Ranked Domain View

In [ ]:
rank_cols = ['primary_hit_at_1', 'primary_hit_at_2', 'primary_hit_at_3', 'primary_hit_at_5']
rank_plot = ranked.set_index('profile')[rank_cols].T
ax = rank_plot.plot(marker='o', figsize=(8, 4))
ax.set_title('Primary-domain ranked hit rate')
ax.set_xlabel('Cutoff')
ax.set_ylabel('Hit rate')
ax.set_ylim(0, 1.05)
ax.set_xticks(range(len(rank_cols)))
ax.set_xticklabels(['1', '2', '3', '5'])
plt.tight_layout()

## Safety Signal Diagnostics

In [ ]:
signal_f1 = safety_signal_per_label.pivot(index='safety_signal', columns='profile', values='f1')
ax = signal_f1.plot(kind='barh', figsize=(9, 4))
ax.set_title('Safety-signal F1 by signal')
ax.set_xlabel('F1')
ax.set_ylabel('Safety signal')
ax.set_xlim(0, 1.05)
plt.tight_layout()

display(safety_signal_per_label.sort_values(['profile', 'f1']))

## Working Decision

The classification ladder is now measurable without training. The next experimental rung should stay dev-only and lightweight: add a transparent embedding or nearest-neighbor classifier candidate, then leave any local LLM JSON classifier or fine-tuning run as an explicit heavy run.